# Compute chunk c04_cpu_t0 (order 4) — generated by `scripts/plan_chunks.py`; do not edit

**Tier 0:** pre-stage-2, dev split only (smoke, pilot). **Window:** now - 25 Oct (Gate 1 on 18 Oct; panel freeze 25 Oct). **Estimate:** 8.52–8.52 h (RL-2026-10-01-06 CPU benchmark x 3.7 (params_b ratio, configs/models.yaml), worst case); quota cost x1 0–0 h (the x1 / x2 readings of a 2xT4 hour: `docs/COMPUTE_PLAN.md`).

| run line | models | run-directory tag |
|---|---|---|
| smoke_20 | phogpt-4b-chat | cpu |
| pilot_t1_200 | phogpt-4b-chat | cpu |
| pilot_xcopa_200 | phogpt-4b-chat | cpu |

Resumable: every job runs `scripts/run_eval.py --resume`; outputs are pushed to the private runs dataset every `PUSH_EVERY_MINUTES` and after each model; `scripts/check_run.py` writes `stats.json` and `results_hashes.json` per run directory. If the session ends early, attach the runs dataset and run this notebook again: finished rows are skipped. The chunk is done when every job reports `ok`. Below is `notebooks/kaggle_cpu_pilot.ipynb` with its parameters cell preset (`JOBS`, `VERIFY_ITEM_KEYS`, `RUN_LABEL`).

# NóiLái — Kaggle CPU: the first real run and the Gate 1 pilot (no GPU quota)

Runs panel models on a Kaggle **CPU** session through the Hugging Face backend in float32
(`--backend hf --device cpu --dtype float32`): first the 20-item smoke line (`smoke_20`, the first real
run: harness → scoring → statistics → hash gate), then, with `MODE = "pilot"`, the two Gate 1 pilot lines
of `configs/run_plan.yaml` (`pilot_t1_200`, nfc vs nfd; `pilot_xcopa_200`), with the EXPLORATORY T1
forced choice (`docs/FORCED_CHOICE_EXPLORATORY.md`). A CPU session uses no GPU hours; it is slower, so
run one model per session (`MODELS`). What a CPU run cannot give is said in `docs/COMPUTE_PLAN.md` (no GPU
tokens/s, which DD 8.5 needs to re-price the GPU lines: a short GPU smoke does that).

Attach before starting: accelerator **None** (CPU); internet on; Kaggle Secrets `HF_TOKEN` (an HF account
that has accepted the Gemma licence for gemma-3-1b-it), `GITHUB_TOKEN` only without a bundle,
`KAGGLE_USERNAME` + `KAGGLE_KEY` for the dataset push; the private datasets `noilai-bundle` and
`noilai-release` (the frozen `data/release/`), and from the second session on `noilai-runs`.

Cells: (a) environment · (b) clone · (b') restore · (c) install (CPU) · (c') pin this session's models ·
(d) resources + item-file gate · (e) **RUN** first run (20 items) · (f) **RUN** pilot · (g) outputs → dataset ·
(h) compute log (CPU rows: zero GPU-hours).

In [ ]:
# ---- parameters: edit this cell only; nothing below asks for input ------------------------------
import time
SESSION_T0 = time.time()

REPO_URL = "https://github.com/BEEEEEEBAGON/noilai.git"
REPO_REF = "main"                                   # pin a commit hash for a recorded run
REPO_SUBDIR = "noilai"                              # both layouts resolve (the clone cell checks for pyproject.toml)
BUNDLE_PATH = "/kaggle/input/noilai-bundle/noilai-main.bundle"
CLONE_DIR = "/kaggle/working/repo"
WORK_DIR = "/kaggle/working"
HF_HOME = "/kaggle/tmp/hf"
ITEMS_DATASET_DIR = "/kaggle/input/noilai-release"
RUNS_RESTORE_DIR = "/kaggle/input/noilai-runs"      # None on the very first session
VERIFY_ITEM_KEYS = None

GITHUB_TOKEN_SECRET = "GITHUB_TOKEN"
HF_TOKEN_SECRET = "HF_TOKEN"

MODE = 'pilot'   # chunk c04_cpu_t0 (scripts/plan_chunks.py)
MODELS = ['phogpt-4b-chat']   # chunk c04_cpu_t0 (scripts/plan_chunks.py)
RUN_IDS = ['smoke_20', 'pilot_t1_200', 'pilot_xcopa_200']   # chunk c04_cpu_t0 (scripts/plan_chunks.py)
FORCED_CHOICE = True   # chunk c04_cpu_t0 (scripts/plan_chunks.py)
PIN_MODELS = True                # (c') resolve and apply revision hashes before running (DD 7.1)
ALLOW_UNPINNED_REVISION = False  # True only for a rehearsal with PIN_MODELS = False
EXTRA_ARGS = ""                  # further run_eval.py flags, one shell-quoted string
ACCOUNT_HOLDER_ROLE = ""         # ROLE of the account holder (DD 11.2), never a name
MIN_RAM_GB = 12                  # fp32 weights: ~4 GB for a 1B, ~8 GB for a 2B, ~16 GB for a 4B model, plus activations
TRANSFORMERS_OVERRIDE = None     # e.g. "4.46.3" for phogpt-4b-chat if its MPT remote code fails under the pinned 5.x (DD 7.2) [UNCERTAIN: verify the version]; recorded in the environment file
PUSH_EVERY_MINUTES = 30
DRY_RUN = False

PLATFORM, GPU_TYPE, N_GPUS = "kaggle", "cpu", 0
SESSION_HARDWARE = "cpu"
MAX_SESSION_HOURS = 10.0         # launch no new model or line past 10 h of a 12-h session [UNCERTAIN: verify] the CPU session limit and RAM (~30 GB) in Kaggle's settings
RUN_LABEL = 'c04_cpu_t0'   # chunk c04_cpu_t0 (scripts/plan_chunks.py)
PUSH_DATASET = True
PUSH_AFTER_EACH_MODEL = True
KAGGLE_DATASET_SLUG = "noilai-runs"
CREATE_DATASET = False

In [ ]:
# secrets: environment -> Colab userdata -> Kaggle Secrets. Values go into os.environ for the
# library that needs them and are reported as found / not found. Nothing here prints a value.
import os

def _secret(name):
    val = os.environ.get(name)
    if val:
        return val
    try:
        from google.colab import userdata          # Colab
        try:
            val = userdata.get(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    try:
        from kaggle_secrets import UserSecretsClient   # Kaggle
        try:
            val = UserSecretsClient().get_secret(name)
        except Exception:
            val = None
        if val:
            return val
    except ImportError:
        pass
    return None

def _export(name, required=False):
    val = _secret(name)
    if val:
        os.environ[name] = val
        print(f"secret {name}: found ({len(val)} chars, not shown)")
        return True
    print(f"secret {name}: not found")
    if required:
        raise SystemExit(f"{name} is required for this notebook")
    return False

In [ ]:
# (a) environment check (CPU session): interpreter, cores, memory, disk; no GPU is used or required
import json, os, shutil, sys
print(sys.version.split()[0], "on", sys.platform)
ENV_CHECK = {"cpus": os.cpu_count(), "ram_gb": None}
try:
    with open("/proc/meminfo") as fh:
        ENV_CHECK["ram_gb"] = round(int(fh.readline().split()[1]) / 2**20, 1)
except OSError:
    pass
du = shutil.disk_usage(WORK_DIR)
print(f"disk at {WORK_DIR}: {du.free / 2**30:.1f} GB free of {du.total / 2**30:.1f} GB")
print(json.dumps(ENV_CHECK))
if ENV_CHECK["ram_gb"] and ENV_CHECK["ram_gb"] < MIN_RAM_GB:
    raise SystemExit(f"{ENV_CHECK['ram_gb']} GB of RAM: below MIN_RAM_GB ({MIN_RAM_GB}) for the largest model in MODELS (fp32 weights)")

In [ ]:
# (b) clone: from a git bundle in a private dataset / Drive when present (no token needed), else from
# GitHub with a token from the platform's secret store. The token reaches git only through GIT_ASKPASS,
# a two-line helper that echoes an environment variable: it is never in a URL, an argv or this output.
import os, stat, subprocess, sys
from pathlib import Path

clone_dir = Path(CLONE_DIR)
bundle = Path(BUNDLE_PATH) if BUNDLE_PATH else None
if bundle is not None and not bundle.exists():
    print(f"bundle {bundle} not found; falling back to {REPO_URL}")
    bundle = None
askpass = None
if bundle is None:
    if "OWNER/REPO" in REPO_URL:
        raise SystemExit("set REPO_URL (or provide BUNDLE_PATH) in the parameters cell")
    if _export(GITHUB_TOKEN_SECRET):
        askpass = Path.home() / ".noilai_askpass.sh"
        askpass.write_text("#!/bin/sh\necho \"$" + GITHUB_TOKEN_SECRET + "\"\n")
        askpass.chmod(stat.S_IRWXU)
# the environment git runs in is built AFTER the export, so the askpass helper finds the token in it
env = dict(os.environ, GIT_TERMINAL_PROMPT="0")
if askpass is not None:
    env["GIT_ASKPASS"] = str(askpass)
source = str(bundle) if bundle else REPO_URL
if (clone_dir / ".git").exists():
    subprocess.run(["git", "-C", str(clone_dir), "fetch", "--tags", source, "+refs/heads/*:refs/remotes/origin/*"], check=True, env=env)
else:
    subprocess.run(["git", "clone", source, str(clone_dir)], check=True, env=env)
# a persisted clone may hold a STALE local branch named like REPO_REF: try the freshly fetched
# remote-tracking ref first and the literal ref last (commit hashes and tags have no remote form)
import re
cands = [REPO_REF] if re.fullmatch(r"[0-9a-f]{7,40}", REPO_REF) else [f"origin/{REPO_REF}", REPO_REF]
for cand in cands:
    if subprocess.run(["git", "-C", str(clone_dir), "checkout", "--detach", cand], env=env).returncode == 0:
        break
else:
    raise SystemExit(f"cannot check out {REPO_REF} (tried {cands})")
HEAD = subprocess.run(["git", "-C", str(clone_dir), "rev-parse", "HEAD"], check=True, env=env,
                      capture_output=True, text=True).stdout.strip()
PROJECT = clone_dir / REPO_SUBDIR if (clone_dir / REPO_SUBDIR / "pyproject.toml").exists() else clone_dir
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT / "scripts"))
print("HEAD", HEAD)
print("project", PROJECT)

In [ ]:
# (b') restore: the clone is ephemeral, so the runs tree persisted by the previous session (the private
# Kaggle dataset `noilai-runs` attached as input, or the Drive folder the API notebook copies to) is
# copied back into PROJECT/data/runs BEFORE anything runs: outputs.jsonl / manifest.json / unchanged.jsonl
# per run directory (so `--resume` continues at the first undone row instead of item 0), api_ledger.json
# (so a day's quota is not re-spent on rows already answered; DD 7.3 "a hit never re-calls") and the
# compute log (so the session's rows append to the campaign's). Nothing here overwrites a newer local file.
import shutil
from pathlib import Path
RESTORED = {"from": None, "run_dirs": 0, "ledger": False, "compute_log": False}
_src = Path(RUNS_RESTORE_DIR) if RUNS_RESTORE_DIR else None
if _src is not None and _src.exists():
    runs_src = _src / "runs" if (_src / "runs").is_dir() else _src        # the dataset holds runs/ + compute_log.csv, or the runs tree itself
    runs_dst = PROJECT / "data" / "runs"
    runs_dst.mkdir(parents=True, exist_ok=True)
    for child in sorted(runs_src.iterdir()):
        if child.name in ("__pycache__", ".ipynb_checkpoints"):
            continue
        if child.is_dir():
            shutil.copytree(child, runs_dst / child.name, dirs_exist_ok=True)
            RESTORED["run_dirs"] += 1
        elif child.name == "api_ledger.json":
            shutil.copy2(child, runs_dst / child.name)
            RESTORED["ledger"] = True
        elif child.suffix in (".json", ".jsonl", ".txt", ".csv"):
            shutil.copy2(child, runs_dst / child.name)
    for cand in (_src / "compute_log.csv", runs_src / "compute_log.csv"):
        if cand.exists() and not (PROJECT / "data" / "compute_log.csv").exists():
            shutil.copy2(cand, PROJECT / "data" / "compute_log.csv")
            RESTORED["compute_log"] = True
            break
    RESTORED["from"] = str(runs_src)
    print("restored", RESTORED)
else:
    print("nothing to restore (first session, or RUNS_RESTORE_DIR", RUNS_RESTORE_DIR, "is not attached): every run starts at item 0")

In [ ]:
# (c) install for the CPU route: the project with its eval extra under the campaign's pins
# (configs/env_pins.txt), WITHOUT the engines (vLLM, llama.cpp) and without re-pinning torch (the image's
# CPU torch is used and recorded). The Hugging Face stack is pinned exactly as for the GPU runs, so a CPU
# result differs from a GPU result only by device arithmetic.
import os, subprocess, sys
from pathlib import Path
_skip = ("vllm", "torch", "llama-cpp-python")
pins = [ln.split("#")[0].strip() for ln in (PROJECT / "configs" / "env_pins.txt").read_text().splitlines()
        if ln.strip() and not ln.startswith("#")]
pins = [p for p in pins if p and p.split("==")[0].strip() not in _skip]
cpath = Path(WORK_DIR) / "cpu_constraints.txt"
cpath.write_text("\n".join(pins) + "\n")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-c", str(cpath), "-e", ".[eval]"], check=True)
if TRANSFORMERS_OVERRIDE:          # the documented PhoGPT fallback (DD 7.2): a 4.x transformers for MPT's remote code
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"transformers=={TRANSFORMERS_OVERRIDE}"], check=True)
    print("transformers overridden to", TRANSFORMERS_OVERRIDE, "(recorded in the environment file)")
RUN_PYTHON = sys.executable
os.environ["HF_HOME"] = HF_HOME
Path(HF_HOME).mkdir(parents=True, exist_ok=True)
_export(HF_TOKEN_SECRET)
if HF_TOKEN_SECRET != "HF_TOKEN" and os.environ.get(HF_TOKEN_SECRET):
    os.environ["HF_TOKEN"] = os.environ[HF_TOKEN_SECRET]
import colab_setup as CS
ENV_RECORD = CS.record_environment(PROJECT / "data" / "runs" / "env")
print("environment recorded at", ENV_RECORD)

In [ ]:
# (c') panel pins for the models of this session (DESIGN_DECISIONS 7.1): resolve each model's Hub revision, licence,
# gating and tokenizer SHA-256 (scripts/pin_panel.py), write the hashes into the clone's configs/models.yaml, verify,
# and keep the manifest with the outputs. A gated model whose licence the HF account has not accepted fails here,
# with the reason, before anything runs. The repository's models.yaml is updated from this manifest by the author
# (the panel-freeze commit), never by the notebook.
import json, shutil, subprocess, sys
from pathlib import Path
if PIN_MODELS:
    r = subprocess.run([sys.executable, "scripts/pin_panel.py", "--names", *MODELS, "--apply"], capture_output=True, text=True, check=False)
    print(r.stdout[-2000:], r.stderr[-2000:])
    man = json.loads((PROJECT / "configs" / "panel_manifest.json").read_text())
    errors = [(e["name"], e.get("error") or e.get("note")) for e in man["entries"] if e["status"] != "pinned"]
    out = Path(WORK_DIR) / "noilai_runs_out"
    out.mkdir(parents=True, exist_ok=True)
    shutil.copy2(PROJECT / "configs" / "panel_manifest.json", out / "panel_manifest.json")
    if errors:
        raise SystemExit(f"not pinned: {errors} (accept the model licence on the Hub with the HF_TOKEN account, or fix the id)")
    v = subprocess.run([sys.executable, "scripts/pin_panel.py", "--verify"], capture_output=True, text=True, check=False)
    print(v.stdout)
    if v.returncode:
        raise SystemExit("pin manifest does not verify")
else:
    print("PIN_MODELS is False: the runs below are rehearsals (ALLOW_UNPINNED_REVISION must be True; manifests record revision null)")

In [ ]:
# (d) third-party resources (SHA-256 against data/HASHES.json) and the item file(s) of the runs this
# notebook will execute: the keys are DERIVED from the run ids (plus the smoke run) unless
# VERIFY_ITEM_KEYS lists them; a seeded sub-sample the plan derives from another file is written first
# (--materialize). Each file is checked for its hash against configs/run_plan.yaml, the header record,
# the canary on every row and its row/cell counts. A failure stops the notebook here.
import shutil, subprocess, sys
from pathlib import Path
import kaggle_run_plan as KRP
r = subprocess.run([sys.executable, "scripts/fetch_resources.py"], check=False)
if r.returncode:
    raise SystemExit("resource fetch or hash check failed (see above)")
if ITEMS_DATASET_DIR:
    src = Path(ITEMS_DATASET_DIR)
    if not src.exists():
        raise SystemExit(f"ITEMS_DATASET_DIR {src} not found: attach the private dataset that holds data/release")
    rel = Path(KRP.load_plan()["release"])            # e.g. data/release/v0.3: the plan's `release` key decides where the files go
    dest = PROJECT / rel
    if (src / rel.name).exists():                      # the dataset holds the data/release tree (one directory per version)
        shutil.copytree(src / rel.name, dest, dirs_exist_ok=True)
    elif (src / "manifest.json").exists():             # the dataset holds ONE release directory
        shutil.copytree(src, dest, dirs_exist_ok=True)
    else:
        raise SystemExit(f"{src} holds neither {rel.name}/ nor a release manifest.json: attach the frozen release")
    print("release copied from", src, "to", dest)
_run_ids = list(globals().get("RUN_IDS") or [globals().get("RUN_ID")]) + list(globals().get("SMOKE_RUN_IDS") or [])
KEYS = VERIFY_ITEM_KEYS or KRP.item_keys_for_runs(KRP.load_plan(), [r for r in _run_ids if r])
print("item files to verify:", KEYS)
for key in KEYS:
    r = subprocess.run([sys.executable, "scripts/kaggle_verify_items.py", "--key", key, "--materialize"], check=False)
    if r.returncode:
        raise SystemExit(f"item file {key} failed verification; nothing runs on it")

In [ ]:
# outputs hand-off used after EVERY model (DD 7.6: Kaggle keeps /kaggle/working only if the version
# completes, so a session that times out must already have pushed what it finished) and once more at
# the end: copy data/runs (+ compute log) into /kaggle/working and add a version to a private Kaggle
# dataset through scripts/kaggle_dataset.py (credentials from Secrets, never printed).
import os, shutil, subprocess, sys
from pathlib import Path

def push_outputs(label):
    out = Path(WORK_DIR) / "noilai_runs_out"
    shutil.copytree(PROJECT / "data" / "runs", out / "runs", dirs_exist_ok=True)
    if (PROJECT / "data" / "compute_log.csv").exists():
        shutil.copy2(PROJECT / "data" / "compute_log.csv", out / "compute_log.csv")
    print("outputs copied to", out)
    if not PUSH_DATASET:
        return
    for name in ("KAGGLE_USERNAME", "KAGGLE_KEY"):
        _export(name)
    if os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY"):
        cmd = [sys.executable, "scripts/kaggle_dataset.py", "--src", str(PROJECT / "data" / "runs"),
               "--src", str(PROJECT / "data" / "compute_log.csv"), "--dest", str(Path(WORK_DIR) / "noilai_runs_dataset"),
               "--slug", KAGGLE_DATASET_SLUG, "--message", f"{label} @ {HEAD[:8]}"]
        if CREATE_DATASET and not globals().get("_DATASET_CREATED"):
            cmd.append("--create")
        rc = subprocess.run(cmd, check=False).returncode
        if rc == 0:
            globals()["_DATASET_CREATED"] = True
    else:
        print("Kaggle credentials not found: dataset push skipped (outputs are still under", out, ")")

def after_each_model(res):
    if PUSH_AFTER_EACH_MODEL:
        push_outputs(f"{res['run']}/{res['model']} {res['status']}")

In [ ]:
# after each RUN cell: scripts/check_run.py on every finished run directory: item gate, deterministic rescoring,
# stats.json (accuracy with the base-pair cluster-bootstrap CI, the copy-baseline gain of PREREG section 10, the paired
# NFD contrast, the EXPLORATORY T1 forced choice against chance) and results_hashes.json
import json, subprocess, sys
from pathlib import Path

def check_runs(results):
    summaries = []
    for r in results:
        run_dir = Path(r["out"]) if r.get("out") else PROJECT / "data" / "runs" / f"{r['run']}__{r['model']}"   # "out" carries a chunk's __<tag>
        if not (run_dir / "scores.jsonl").exists():
            print(r["run"], r["model"], "->", r["status"], "(no scores: not checked)")
            continue
        c = subprocess.run([sys.executable, "scripts/check_run.py", "--run", str(run_dir)], capture_output=True, text=True, check=False)
        print(c.stdout.strip() or c.stderr[-1500:])
        st = json.loads((run_dir / "stats.json").read_text()) if (run_dir / "stats.json").exists() else {}
        summaries.append({"run": r["run"], "model": r["model"], "status": r["status"],
                          "by_task_arm": {k: (round(v["accuracy"], 3), [round(x, 3) for x in v["ci"]]) for k, v in st.get("by_task_arm", {}).items()},
                          "t1_forced_choice": {k: (round(v["accuracy"], 3), round(v["chance"], 3)) for k, v in st.get("t1_forced_choice", {}).items()}})
    print(json.dumps(summaries, ensure_ascii=False, indent=1))
    return summaries

### RUN CELL (e) — the first real run: 20 items (`smoke_20`)
The plan's smoke line (the only kind of line allowed a `--limit`) on each model of `MODELS`, through the HF
backend on CPU, then `scripts/check_run.py` on the result. A failure stops the notebook.

In [ ]:
# RUN CELL (e): the first real run. The CPU route = the HF backend in float32 on a session with no accelerator
# (configs carry t4 hardware, hence the documented mismatch flag; the compute log records device "cpu" x 0).
import shlex
import kaggle_run_plan as KRP
CPU_EXTRA = ["--backend", "hf", "--device", "cpu", "--dtype", "float32"]
RUN_DIR_TAG = "cpu"      # data/runs/<run>__<model>__cpu: a T4 run of the same line keeps its own directory, so the first
                         # GPU smoke really runs and its outputs can be compared with these row by row
RUN_EXTRA = CPU_EXTRA + shlex.split(EXTRA_ARGS) + (["--account-holder", ACCOUNT_HOLDER_ROLE] if ACCOUNT_HOLDER_ROLE else [])
if not ACCOUNT_HOLDER_ROLE:
    print("WARNING: ACCOUNT_HOLDER_ROLE is empty; the manifests will record account_holder 'unknown' (DD 11.2 asks for the role)")
SMOKE_RESULTS = KRP.execute("smoke_20", models=MODELS, platform=PLATFORM, dry_run=DRY_RUN, continue_on_error=False,
                            extra=RUN_EXTRA, python=RUN_PYTHON, session_hardware=SESSION_HARDWARE,
                            allow_hardware_mismatch=True, session_t0=SESSION_T0, max_session_hours=MAX_SESSION_HOURS,
                            after_each=after_each_model, allow_unpinned_revision=ALLOW_UNPINNED_REVISION,
                            chunk_tag=RUN_DIR_TAG)
FIRST_RUN = check_runs(SMOKE_RESULTS)
if any(str(r["status"]).startswith(("failed", "skipped", "refused")) for r in SMOKE_RESULTS):
    raise SystemExit("the first run failed or was skipped: read the status above before the pilot cell")

### RUN CELL (f) — the Gate 1 pilot lines (`MODE = "pilot"`)
`pilot_t1_200` (200 dev items, 50 per variant, nfc vs nfd, p0, three demonstrations) with the exploratory
forced choice, and `pilot_xcopa_200` (log-likelihood), then `check_run.py`. The go/no-go itself is decided
from these files by the rule of PREREGISTRATION §10, over the three pilot models together; this cell only
prints each model's numbers.

In [ ]:
# RUN CELL (f): the pilot lines, one model at a time, resumable (--resume via the driver); a background push
# every PUSH_EVERY_MINUTES keeps a killed session's rows.
import threading
RESULTS = []
if MODE == "pilot":
    _stop_push = threading.Event()

    def _periodic_push():
        while not _stop_push.wait(PUSH_EVERY_MINUTES * 60):
            try:
                push_outputs("periodic")
            except Exception as e:
                print("periodic push failed:", repr(e))
    _pusher = threading.Thread(target=_periodic_push, daemon=True)
    _pusher.start()
    try:
        for run_id in [r for r in RUN_IDS if r != "smoke_20"]:
            extra = RUN_EXTRA + (["--t1-forced-choice"] if FORCED_CHOICE and run_id.startswith("pilot_t1") else [])
            RESULTS += KRP.execute(run_id, models=MODELS, platform=PLATFORM, dry_run=DRY_RUN, continue_on_error=True,
                                   extra=extra, python=RUN_PYTHON, session_hardware=SESSION_HARDWARE,
                                   allow_hardware_mismatch=True, session_t0=SESSION_T0,
                                   max_session_hours=MAX_SESSION_HOURS, after_each=after_each_model,
                                   allow_unpinned_revision=ALLOW_UNPINNED_REVISION, chunk_tag=RUN_DIR_TAG)
    finally:
        _stop_push.set()
        _pusher.join(timeout=5)
    PILOT = check_runs(RESULTS)
else:
    print("MODE is", MODE, "- pilot lines not run")

In [ ]:
# (g) final hand-off: the same push as after each model, once more for the session as a whole.
push_outputs(RUN_LABEL)

In [ ]:
# (h) GPU-hours log (checklist C1). kaggle_run_plan wrote one row per executed model run; this adds the
# session's remaining wall time (startup, install, model downloads) as "overhead" and prints the totals.
import shutil
import compute_log as CL
LOG = PROJECT / "data" / "compute_log.csv"
run_hours = sum((r.get("hours") or 0.0) for r in (globals().get("RESULTS", []) + globals().get("SMOKE_RESULTS", [])))
overhead = max(0.0, CL.hours_since(SESSION_T0) - run_hours)
CL.append_entry(LOG, CL.Entry(date=CL.today(), platform=PLATFORM, gpu_type=GPU_TYPE, n_gpus=N_GPUS,
                              hours=overhead, run_id=f"session-{RUN_LABEL}", purpose="overhead"))
print(CL.format_totals(CL.totals(CL.read_entries(LOG))))
out = Path(WORK_DIR) / "noilai_runs_out"
out.mkdir(parents=True, exist_ok=True)
shutil.copy2(LOG, out / "compute_log.csv")
print("log copied to", out / "compute_log.csv", "-> merge into the repository's data/compute_log.csv")